In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta
import os

fake = Faker('en_IN')  # Indian locale
random.seed(42)
np.random.seed(42)

print("✅ Libraries loaded successfully")

✅ Libraries loaded successfully


In [2]:
UPI_HANDLES = ['@okicici', '@okhdfcbank', '@oksbi', '@okaxis',
               '@paytm', '@ybl', '@upi', '@ibl', '@kotak']

def make_vpa(name):
    clean = name.lower().replace(' ', '.').replace("'", '')
    return clean + str(random.randint(1, 999)) + random.choice(UPI_HANDLES)

accounts = []
n_accounts = 5000
n_mules = int(n_accounts * 0.10)  # 500 mule accounts

for i in range(n_accounts):
    name = fake.name()
    accounts.append({
        'account_id':       f'ACC{i:05d}',
        'vpa':              make_vpa(name),
        'name':             name,
        'bank':             random.choice(['SBI','HDFC','ICICI','Axis','Kotak']),
        'account_age_days': random.randint(30, 1825),
        'is_mule':          1 if i < n_mules else 0,
    })

df_accounts = pd.DataFrame(accounts)

print(f"✅ Accounts created : {len(df_accounts)}")
print(f"✅ Mule accounts    : {df_accounts['is_mule'].sum()}")
print(f"\nSample VPAs:")
print(df_accounts['vpa'].head(5).to_string())

✅ Accounts created : 5000
✅ Mule accounts    : 500

Sample VPAs:
0    upasna.shroff655@okhdfcbank
1           samuel.tak282@okaxis
2     ishani.divan755@okhdfcbank
3           vrinda.sunder605@upi
4          yamini.datta96@okaxis


In [3]:
df_accounts

,account_id,vpa,name,bank,account_age_days,is_mule
0,ACC00000,upasna.shroff655@okhdfcbank,Upasna Shroff,SBI,1548,1
1,ACC00001,samuel.tak282@okaxis,Samuel Tak,HDFC,315,1
2,ACC00002,ishani.divan755@okhdfcbank,Ishani Divan,Kotak,208,1
3,ACC00003,vrinda.sunder605@upi,Vrinda Sunder,SBI,91,1
4,ACC00004,yamini.datta96@okaxis,Yamini Datta,HDFC,1064,1
...,...,...,...,...,...,...
4995,ACC04995,indrajit.varghese835@upi,Indrajit Varghese,HDFC,878,0
4996,ACC04996,neelima.bhargava485@okicici,Neelima Bhargava,ICICI,788,0
4997,ACC04997,daksh.aggarwal122@paytm,Daksh Aggarwal,HDFC,1026,0
4998,ACC04998,dipta.palla901@ibl,Dipta Palla,Axis,1235,0


In [4]:
start_date = datetime(2024, 1, 1)
all_txns = []

# Hour weights — UPI peaks at morning (10am-12pm) and evening (7pm-9pm)
hour_weights = [0.2, 0.1, 0.1, 0.1, 0.2, 0.5, 1.0, 1.5, 2.0, 2.5,
                3.0, 2.8, 2.5, 2.2, 2.0, 2.3, 2.5, 2.8, 3.2, 3.5,
                3.0, 2.5, 1.5, 0.8]

for i in range(50000):
    sender   = df_accounts.sample(1).iloc[0]
    receiver = df_accounts.sample(1).iloc[0]

    # Make sure sender and receiver are different accounts
    while receiver['account_id'] == sender['account_id']:
        receiver = df_accounts.sample(1).iloc[0]

    # Random timestamp over 90 days
    day_offset = random.randint(0, 89)
    hour       = random.choices(range(24), weights=hour_weights)[0]
    minute     = random.randint(0, 59)
    timestamp  = start_date + timedelta(days=day_offset, hours=hour, minutes=minute)

    # Log-normal amount — realistic UPI distribution (~Rs.850 average)
    amount = round(np.random.lognormal(mean=6.5, sigma=1.2), 2)
    amount = min(amount, 100000)  # UPI per-transaction limit

    txn_type = random.choices(
        ['P2P', 'MERCHANT', 'BILL'],
        weights=[60, 30, 10]
    )[0]

    all_txns.append({
        'txn_id':       f'TXN{i:07d}',
        'sender_id':    sender['account_id'],
        'sender_vpa':   sender['vpa'],
        'receiver_id':  receiver['account_id'],
        'receiver_vpa': receiver['vpa'],
        'amount':       amount,
        'timestamp':    timestamp,
        'txn_type':     txn_type,
        'is_fraud':     0,
    })

df_txns = pd.DataFrame(all_txns).sort_values('timestamp').reset_index(drop=True)

print(f"✅ Transactions generated : {len(df_txns)}")
print(f"✅ Date range : {df_txns['timestamp'].min()} → {df_txns['timestamp'].max()}")
print(f"\nSample transactions:")
df_txns[['txn_id','sender_vpa','amount','txn_type','timestamp']].head(3)

✅ Transactions generated : 50000
✅ Date range : 2024-01-01 00:11:00 → 2024-03-30 23:58:00

Sample transactions:


,txn_id,sender_vpa,amount,txn_type,timestamp
0,TXN0035624,udant.sharaf35@okaxis,308.60,P2P,2024-01-01 00:11:00
1,TXN0001686,gautam.dara396@okaxis,9481.11,P2P,2024-01-01 00:33:00
2,TXN0045757,chameli.chandran672@okhdfcbank,195.22,P2P,2024-01-01 00:44:00


In [5]:
df_txns['timestamp'] = pd.to_datetime(df_txns['timestamp'])

# Pattern A — Mule accounts
mule_ids = df_accounts[df_accounts['is_mule'] == 1]['account_id'].tolist()
mule_txn_idx = df_txns[df_txns['receiver_id'].isin(mule_ids)].index
df_txns.loc[mule_txn_idx[:2000], 'is_fraud'] = 1

# Pattern B — Velocity spike (SIM swap)
for acc in df_accounts.sample(30)['account_id']:
    mask = df_txns['sender_id'] == acc
    spike_idx = df_txns[mask].head(25).index
    df_txns.loc[spike_idx, 'timestamp'] = df_txns.loc[spike_idx[0], 'timestamp']
    df_txns.loc[spike_idx, 'is_fraud'] = 1

# Pattern C — Amount just under ₹1 lakh limit
round_mask = (df_txns['amount'] >= 99000) & (df_txns['amount'] <= 99999)
df_txns.loc[round_mask, 'is_fraud'] = 1

# Results
fraud_rate = df_txns['is_fraud'].mean() * 100
print(f"✅ Fraud rate         : {fraud_rate:.2f}%")
print(f"✅ Fraud transactions : {df_txns['is_fraud'].sum()}")

# Save
os.makedirs('../data/raw', exist_ok=True)
df_txns.to_csv('../data/raw/upi_synthetic.csv', index=False)
df_accounts.to_csv('../data/raw/upi_accounts.csv', index=False)
print("✅ Saved to data/raw/")

✅ Fraud rate         : 4.57%
✅ Fraud transactions : 2283
✅ Saved to data/raw/


In [6]:
print(f"Total transactions  : {len(df_txns)}")
print(f"Fraud transactions  : {df_txns['is_fraud'].sum()}")
print(f"Legit transactions  : {(df_txns['is_fraud']==0).sum()}")
print(f"Fraud rate          : {df_txns['is_fraud'].mean()*100:.2f}%")
print()

# Breakdown by fraud pattern
print("--- Fraud breakdown by pattern ---")
print(f"Pattern A (mule)    : transactions to mule accounts")
print(f"Pattern B (velocity): {df_txns[df_txns['is_fraud']==1]['sender_id'].nunique()} unique senders flagged")
print(f"Pattern C (limit)   : {((df_txns['amount']>=99000)&(df_txns['amount']<=99999)).sum()} near-limit txns")
print()

# Check files saved correctly
import os
for f in ['upi_synthetic.csv', 'upi_accounts.csv']:
    path = f'../data/raw/{f}'
    size = os.path.getsize(path) / 1024
    print(f"✅ {f} — {size:.1f} KB")

Total transactions  : 50000
Fraud transactions  : 2283
Legit transactions  : 47717
Fraud rate          : 4.57%

--- Fraud breakdown by pattern ---
Pattern A (mule)    : transactions to mule accounts
Pattern B (velocity): 1678 unique senders flagged
Pattern C (limit)   : 0 near-limit txns

✅ upi_synthetic.csv — 5370.8 KB
✅ upi_accounts.csv — 281.8 KB


In [7]:
# Load PaySim — takes 10-20 seconds (6.3M rows)
df_paysim = pd.read_csv('../data/raw/PS_20174392719_1491204439457_log.csv')

print(f"✅ PaySim loaded")
print(f"Shape              : {df_paysim.shape}")
print(f"Fraud rate         : {df_paysim['isFraud'].mean()*100:.4f}%")
print(f"Fraud transactions : {df_paysim['isFraud'].sum()}")
print(f"\nColumns: {df_paysim.columns.tolist()}")
print(f"\nTransaction types:\n{df_paysim['type'].value_counts()}")
df_paysim.head(3)

✅ PaySim loaded
Shape              : (6362620, 11)
Fraud rate         : 0.1291%
Fraud transactions : 8213

Columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']

Transaction types:
type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0


In [8]:
# Keep only relevant columns
df_paysim_clean = df_paysim[[
    'type',           # transaction type
    'amount',         # amount
    'oldbalanceOrg',  # sender balance before
    'newbalanceOrig', # sender balance after
    'oldbalanceDest', # receiver balance before
    'newbalanceDest', # receiver balance after
    'isFraud'         # label — this is what you need
]].copy()

# Rename to match your UPI dataset column names
df_paysim_clean.rename(columns={'isFraud': 'is_fraud'}, inplace=True)

# Keep only TRANSFER and CASH_OUT — these are the only types
# that have fraud in PaySim (PAYMENT, DEBIT, CASH_IN never have fraud)
df_paysim_clean = df_paysim_clean[
    df_paysim_clean['type'].isin(['TRANSFER', 'CASH_OUT'])
]

# Add derived features
df_paysim_clean['log_amount']  = np.log1p(df_paysim_clean['amount'])
df_paysim_clean['balance_diff_sender']   = (df_paysim_clean['oldbalanceOrg'] 
                                            - df_paysim_clean['newbalanceOrig'])
df_paysim_clean['balance_diff_receiver'] = (df_paysim_clean['newbalanceDest'] 
                                            - df_paysim_clean['oldbalanceDest'])

# Flag suspicious balance pattern:
# Sender balance drops to exactly 0 after transaction = account drained
df_paysim_clean['account_drained'] = (
    df_paysim_clean['newbalanceOrig'] == 0
).astype(int)

print(f"✅ PaySim cleaned")
print(f"Shape after filter : {df_paysim_clean.shape}")
print(f"Fraud rate now     : {df_paysim_clean['is_fraud'].mean()*100:.2f}%")
print(f"Fraud transactions : {df_paysim_clean['is_fraud'].sum()}")
print(f"\nTransaction types kept:\n{df_paysim_clean['type'].value_counts()}")

✅ PaySim cleaned
Shape after filter : (2770409, 11)
Fraud rate now     : 0.30%
Fraud transactions : 8213

Transaction types kept:
type
CASH_OUT    2237500
TRANSFER     532909
Name: count, dtype: int64


In [9]:
df_paysim_clean.to_csv('../data/raw/paysim_clean.csv', index=False)

print(f"✅ Saved paysim_clean.csv")
print(f"\nYour data/raw/ folder now has:")
for f in os.listdir('../data/raw/'):
    size = os.path.getsize(f'../data/raw/{f}') / (1024*1024)
    print(f"  📄 {f}  —  {size:.1f} MB")

✅ Saved paysim_clean.csv

Your data/raw/ folder now has:
  📄 paysim_clean.csv  —  249.7 MB
  📄 PS_20174392719_1491204439457_log.csv  —  470.7 MB
  📄 train_identity.csv  —  25.3 MB
  📄 train_transaction.csv  —  651.7 MB
  📄 upi_accounts.csv  —  0.3 MB
  📄 upi_synthetic.csv  —  5.2 MB
